# Irradiance and the Poynting vector {#sec-irradiance}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter1/notebooks/irradiance.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter1/notebooks/irradiance.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter1/optolib.py', 'optolib.py')
else:
    sys.path.insert(0, '..')                 # optolib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import optolib as ol
ol.use_style()
c0, eps0, nm, um = ol.c0, ol.eps0, ol.nm, ol.um

*Slides 70–84: magnetic field of the wave, energy density, Poynting vector, average irradiance, spherical waves, power in a
Gaussian beam.*

## Key relations

The magnetic field always accompanies the electric field, with $B_y=E_x/v$ and $v=c/n$ (slide 71). The energy densities of the
two fields are equal, so the total energy density is $\varepsilon_0\varepsilon_rE_x^2$ (slide 73). Energy flows along
$\mathbf k$ with the **Poynting vector** (slides 74–75):

$$
\mathbf S=v^2\varepsilon_0\varepsilon_r\,\mathbf E\times\mathbf B,\qquad
I=\langle S\rangle=\tfrac12 v\,\varepsilon_0\varepsilon_rE_0^2=\tfrac12 c\,\varepsilon_0nE_0^2 .
$$

- Detectors are far too slow to follow the optical oscillation (≈ 5 fs period at 1550 nm), so they always measure the
  **average irradiance** $I$ (slide 76).
- A **spherical wave** spreads its power over $4\pi r^2$, so $I=P/(4\pi r^2)$ (slides 77–78).
- A **Gaussian beam** concentrates its power: $I_0=2P/(\pi w^2)$ on axis (slides 79–82).

The deep-dive chapter @sec-poynting derives Poynting's theorem step by step and answers the classic objection: *how can energy
flow through vacuum, where there is no current?*

### Field strength from irradiance, live

```{ojs}
//| echo: false
viewof irr = Inputs.form({
  P: Inputs.range([-30, 40], {value: 0, step: 0.5, label: "power (dBm)"}),
  w: Inputs.range([0.1, 5000], {value: 5.2, step: 0.1, label: "beam radius w (µm)"}),
  n: Inputs.range([1, 4], {value: 1.444, step: 0.001, label: "refractive index n"})
})
```

```{ojs}
//| echo: false
{
  const P = 1e-3 * 10 ** (irr.P / 10), w = irr.w * 1e-6;
  const I0 = 2 * P / (Math.PI * w * w), E0 = Math.sqrt(2 * I0 / (299792458 * 8.854e-12 * irr.n));
  const sun = 1000;
  return html`<div class="result-box">P = ${P >= 1 ? P.toFixed(2) + " W" : (P*1e3).toPrecision(3) + " mW"} ·
  peak irradiance I₀ = 2P/(πw²) = <b>${I0.toExponential(2)} W/m²</b> = ${(I0/1e4).toExponential(2)} W/cm²
  (${(I0/sun).toExponential(1)} × sunlight)<br>
  peak field amplitude E₀ = √(2I₀/(cε₀n)) = <b>${E0 >= 1e6 ? (E0/1e6).toPrecision(3) + " MV/m" : (E0/1e3).toPrecision(3) + " kV/m"}</b></div>`
}
```

## Worked example: how strong is the optical field in a fiber and in a chip?

Using $I=\tfrac12c\,\varepsilon_0nE_0^2$ and, for a Gaussian mode, $I_0=2P/(\pi w^2)$:

In [ ]:
def E0_from_I(I, n): return np.sqrt(2*I/(c0*eps0*n))

cases = [("sunlight at ground (1 kW/m²)", None, 1e3, 1.0),
         ("5 mW He-Ne laser (w = 0.5 mm, slide 83)", (5e-3, 0.5e-3), None, 1.0),
         ("1 mW in SMF-28 (w = 5.2 µm)", (1e-3, um(5.2)), None, 1.444),
         ("100 mW in SMF-28", (100e-3, um(5.2)), None, 1.444),
         ("100 mW in a Si wire (A_eff ≈ 0.1 µm²)", None, 100e-3/0.1e-12, 3.48)]
for name, gauss, I, nn in cases:
    if gauss:
        P, w = gauss; I = 2*P/(np.pi*w**2)
    print(f"{name:42s} I = {I:9.2e} W/m²   E0 = {E0_from_I(I, nn):9.2e} V/m")

The silicon-wire case reaches ~$10^{12}$ W/m² (0.1 GW/cm²) with only 100 mW. This is why two-photon absorption and
free-carrier effects appear in silicon photonic circuits at modest powers, and why fiber nonlinearities (Raman, Brillouin;
@sec-scattering) matter over long distances.

## Spherical waves and the inverse-square law (slides 77–78)

In [ ]:
#| label: fig-inverse-square
#| fig-cap: "Irradiance versus distance for a 100 W isotropic bulb (spherical wave, ∝ 1/r²) and a 5 mW He-Ne laser beam (Gaussian, nearly constant within the Rayleigh range, then ∝ 1/z²)."
r = np.logspace(-1, 3, 300)
I_bulb = 100/(4*np.pi*r**2)
w, z0 = ol.gaussian_w(r, 0.5e-3, nm(633))
I_laser = 2*5e-3/(np.pi*w**2)
fig, ax = plt.subplots()
ax.loglog(r, I_bulb, label="100 W bulb, I = P/4πr²")
ax.loglog(r, I_laser, label="5 mW He-Ne, on-axis I = 2P/πw²")
ax.axvline(z0, color=ol.INK2, ls=":", lw=1); ax.text(z0*1.1, 1e-3, "z₀", color=ol.INK2)
ax.set(xlabel="distance (m)", ylabel="irradiance (W/m²)"); ax.legend()
plt.show()

A 5 mW laser is brighter on its axis than a 100 W bulb at any distance shown here. That is why laser safety
(IEC 60825-1) is formulated in terms of irradiance, not power.

::: {.callout-tip collapse="true"}
## Going deeper
- Slide 76 notes that detectors only measure time-averaged irradiance. Estimate the optical period at 1550 nm and compare it with
  the bandwidth of the fastest photodiodes (Chapter 5). What does a coherent receiver measure instead?
- Radiation pressure: $p=I/c$. Estimate the force of a 1 W beam on a mirror, and relate it to optical tweezers and optomechanics.
- For a guided mode, the power is the integral of $S_z$ over the cross-section. Why does the "effective area" $A_\text{eff}$ used
  in nonlinear optics differ from the geometric core area?
:::

::: {.callout-note collapse="true"}
## Complements
- D. J. Griffiths, *Introduction to Electrodynamics*: chapter on electromagnetic waves (energy and momentum).
- Laser-safety standard IEC 60825-1: maximum permissible exposure values are expressed in irradiance.
- @sec-poynting, the deep-dive chapter of this book, derives Poynting's theorem from scratch.
:::

## Try it yourself

1. The solar constant above the atmosphere is 1361 W/m². What is the amplitude of the electric field of sunlight?
2. A 10 W fiber laser is focused to $w=10$ µm. Compare the field with the breakdown field of air (~3 MV/m).
3. Show that the average energy density of a plane wave, multiplied by its speed $v$, gives $I$.